In [1]:
# extrai_andamentos_com_banco.py
# Usa .env → Oracle → Lista de processos → Scraping de andamentos → Excel

import os
import time
import logging
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException
from dotenv import load_dotenv
from pathlib import Path
from sqlalchemy import create_engine
import oracledb

In [2]:
# =============================================
# 1. CONFIGURAÇÃO
# =============================================
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

ENV_PATH = Path(r"C:\Users\luiz.smorais\Documents\MS\STAGE\STAGE.env")
print("Carregando .env de:", ENV_PATH)

if not ENV_PATH.exists():
    raise FileNotFoundError(f".env não encontrado: {ENV_PATH}")

load_dotenv(dotenv_path=ENV_PATH)

# --- SELENIUM ---
usr = os.getenv("USUARIO")
pwd = os.getenv("SENHA")
org = os.getenv("ORGAO")
driver_path = os.getenv("CHROMEDRIVER_PATH")

# --- ORACLE ---
oracle_user = os.getenv("ORACLE_USER")
oracle_pass = os.getenv("ORACLE_PASS")
oracle_host = os.getenv("ORACLE_HOST")
oracle_port = os.getenv("ORACLE_PORT", "1521")
oracle_service = os.getenv("ORACLE_SERVICE")

# Validação
if not all([usr, pwd, org, driver_path, oracle_user, oracle_pass, oracle_host, oracle_service]):
    raise ValueError("Faltam variáveis no .env")

if not os.path.exists(driver_path):
    raise FileNotFoundError(f"chromedriver não encontrado: {driver_path}")

print(f"Usuário SEI: {usr} | Órgão: {org}")
print(f"Oracle: {oracle_user}@{oracle_host}:{oracle_port}/{oracle_service}")

# =============================================
# 2. CONEXÃO AO BANCO ORACLE (CORRIGIDO ORA-28009)
# =============================================
db_url = f"oracle+oracledb://@{oracle_host}:{oracle_port}/?service_name={oracle_service}"
connect_args = {
    "user": oracle_user,
    "password": oracle_pass,
    "mode": oracledb.SYSDBA  # Necessário para SYS
}

try:
    engine = create_engine(db_url, connect_args=connect_args)
    logging.info("Conexão ao Oracle realizada com sucesso!")
except Exception as e:
    logging.error(f"Falha na conexão Oracle: {e}")
    raise

# =============================================
# 3. OBTÉM PROCESSOS DO BANCO
# =============================================
def obter_processos_do_banco():
    query = """
    SELECT num_processo_sei 
    FROM STAGE.MEC_AQUISICAO 
    WHERE num_processo_sei IS NOT NULL
    """
    try:
        df = pd.read_sql(query, engine)
        processos = df['num_processo_sei'].astype(str).tolist()
        logging.info(f"{len(processos)} processos carregados do banco.")
        return processos
    except Exception as e:
        logging.error(f"Erro ao ler do banco: {e}")
        return []

2025-11-07 15:05:41,024 - INFO - Conexão ao Oracle realizada com sucesso!


Carregando .env de: C:\Users\luiz.smorais\Documents\MS\STAGE\STAGE.env
Usuário SEI: luiz.smorais | Órgão: MS
Oracle: sys@10.1.45.234:1521/freepdb1


In [ ]:
# =============================================
# 4. INICIA NAVEGADOR
# =============================================
def iniciar_navegador():
    service = Service(executable_path=driver_path)
    options = Options()
    options.add_argument("--start-maximized")
    # options.add_argument("--headless")
    driver = webdriver.Chrome(service=service, options=options)
    return driver

driver = iniciar_navegador()
wait = WebDriverWait(driver, 20)

# =============================================
# 5. LOGIN NO SEI
# =============================================
def fazer_login():
    print("Fazendo login no SEI...")
    driver.get("https://sei.saude.gov.br/")
    
    wait.until(EC.presence_of_element_located((By.ID, "txtUsuario")))
    driver.find_element(By.ID, "txtUsuario").send_keys(usr)
    driver.find_element(By.ID, "pwdSenha").send_keys(pwd)
    driver.find_element(By.ID, "selOrgao").send_keys(org)
    driver.find_element(By.ID, "sbmLogin").click()
    
    try:
        wait.until(EC.presence_of_element_located((By.ID, "txtPesquisaRapida")))
        print("Login OK!")
        return True
    except:
        print("Falha no login.")
        return False

# =============================================
# 6. SCRAPING DE ANDAMENTOS (ÚNICA FUNÇÃO ATIVA)
# =============================================
def extrair_andamentos_processo(numero: str) -> list:
    numero = numero.strip()
    if not numero:
        return []

    print(f"  → {numero}", end="")
    andamentos = []

    try:
        campo = wait.until(EC.presence_of_element_located((By.ID, "txtPesquisaRapida")))
        campo.clear()
        campo.send_keys(numero)
        campo.send_keys(Keys.RETURN)
        time.sleep(2)

        iframe = wait.until(EC.frame_to_be_available_and_switch_to_it((By.ID, "ifrVisualizacao")))

        try:
            aba = driver.find_element(By.LINK_TEXT, "Andamentos")
            driver.execute_script("arguments[0].click();", aba)
            time.sleep(1.5)
        except:
            print(" → sem andamentos")
            driver.switch_to.default_content()
            return []

        tabela = wait.until(EC.presence_of_element_located((By.ID, "tblAndamentos")))
        linhas = tabela.find_elements(By.TAG_NAME, "tr")[1:]

        if not linhas:
            print(" → 0 linhas")
            driver.switch_to.default_content()
            return []

        for linha in linhas:
            cols = linha.find_elements(By.TAG_NAME, "td")
            if len(cols) < 5: continue

            doc_gerado = ""
            try:
                link_doc = cols[2].find_element(By.TAG_NAME, "a")
                doc_gerado = link_doc.text.strip()
            except:
                doc_gerado = cols[2].text.strip().split("\n")[0]

            andamentos.append({
                "Número Processo": numero,
                "Data/Hora": cols[0].text.strip(),
                "Usuário": cols[3].text.strip(),
                "Ação": cols[2].text.strip(),
                "Documento Gerado": doc_gerado,
                "Observações": cols[4].text.strip(),
                "Link Processo": driver.current_url
            })

        print(f" → {len(linhas)} andamentos")
        driver.switch_to.default_content()
        return andamentos

    except Exception as e:
        print(f" → ERRO: {e}")
        driver.switch_to.default_content()
        return []

# =============================================
# 7. LOOP PRINCIPAL
# =============================================
def main():
    processos = obter_processos_do_banco()
    if not processos:
        print("Nenhum processo encontrado no banco.")
        return

    if not fazer_login():
        return

    todos_andamentos = []
    for proc in processos:
        andamentos = extrair_andamentos_processo(proc)
        todos_andamentos.extend(andamentos)

    if todos_andamentos:
        df = pd.DataFrame(todos_andamentos)
        arquivo = f"{org}_andamentos_do_banco.xlsx"
        df.to_excel(arquivo, index=False)
        print(f"\nExcel salvo: {arquivo}")

        # Hiperlinks
        try:
            import win32com.client as win32
            excel = win32.Dispatch('Excel.Application')
            wb = excel.Workbooks.Open(Path(arquivo).resolve())
            ws = wb.Worksheets(1)
            ws.Columns.AutoFit()
            ws.Range("A1:G1").Font.Bold = True
            for r in range(2, len(df) + 2):
                ws.Cells(r, 7).Value = f'=HYPERLINK("{df.iloc[r-2]["Link Processo"]}", "Abrir")'
            wb.Save()
            wb.Close()
            excel.Quit()
            print("Hiperlinks OK!")
        except Exception as e:
            print(f"Formatação falhou: {e}")
    else:
        print("Nenhum andamento extraído.")

    driver.quit()
    print("\nCONCLUÍDO!")

# =============================================
# 8. EXECUÇÃO
# =============================================
if __name__ == "__main__":
    main()

2025-11-07 15:06:53,276 - INFO - 815 processos carregados do banco.


Fazendo login no SEI...


NoSuchElementException: Message: no such element: Unable to locate element: {"method":"css selector","selector":"[id="sbmLogin"]"}
  (Session info: chrome=142.0.7444.59); For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#nosuchelementexception
Stacktrace:
	GetHandleVerifier [0x0x7ff6ccaee8e5+80021]
	GetHandleVerifier [0x0x7ff6ccaee940+80112]
	(No symbol) [0x0x7ff6cc87060f]
	(No symbol) [0x0x7ff6cc8c8854]
	(No symbol) [0x0x7ff6cc8c8b1c]
	(No symbol) [0x0x7ff6cc91c927]
	(No symbol) [0x0x7ff6cc8f126f]
	(No symbol) [0x0x7ff6cc91968a]
	(No symbol) [0x0x7ff6cc8f1003]
	(No symbol) [0x0x7ff6cc8b95d1]
	(No symbol) [0x0x7ff6cc8ba3f3]
	GetHandleVerifier [0x0x7ff6ccdadc7d+2960429]
	GetHandleVerifier [0x0x7ff6ccda7f3a+2936554]
	GetHandleVerifier [0x0x7ff6ccdc8977+3070247]
	GetHandleVerifier [0x0x7ff6ccb083ce+185214]
	GetHandleVerifier [0x0x7ff6ccb0fe1f+216527]
	GetHandleVerifier [0x0x7ff6ccaf7b24+117460]
	GetHandleVerifier [0x0x7f